# 07. Model Comparison and Blending
This notebook compares the CatBoost, LightGBM, and XGBoost baselines using aligned out-of-fold predictions. It also evaluates whether blending complementary models improves Balanced Accuracy.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

EXPERIMENTS_DIR = PROJECT_ROOT / "artifacts" / "experiments"

print(f"Project root: {PROJECT_ROOT}")
print(f"Experiments directory: {EXPERIMENTS_DIR}")

Project root: C:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final
Experiments directory: C:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\artifacts\experiments


In [7]:
from itertools import combinations

from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    log_loss,
    roc_auc_score,
)

from src.churn_ml.metrics import (
    evaluate_cross_fitted_thresholds,
    optimize_balanced_accuracy_threshold,
    summarize_threshold_plateau,
)
from src.churn_ml.modeling import load_experiment

## Load Baseline Experiments

All models must use the same dataset version, cross-validation folds, row ordering, and target values. This is required for a valid OOF comparison and blending analysis.

In [3]:
EXPERIMENT_IDS = {
    "CatBoost": "catboost_v0_raw_minimal_cv5",
    "LightGBM": "lightgbm_v0_raw_minimal_cv5",
    "XGBoost": "xgboost_v0_raw_minimal_cv5",
}

experiments = {
    model_name: load_experiment(
        experiment_id=experiment_id,
        experiments_dir=EXPERIMENTS_DIR,
    )
    for model_name, experiment_id in EXPERIMENT_IDS.items()
}

for model_name, experiment in experiments.items():
    print(
        f"{model_name:<10} "
        f"{experiment.result.experiment_id:<35} "
        f"OOF rows={len(experiment.oof_predictions):,}"
    )

CatBoost   catboost_v0_raw_minimal_cv5         OOF rows=10,000
LightGBM   lightgbm_v0_raw_minimal_cv5         OOF rows=10,000
XGBoost    xgboost_v0_raw_minimal_cv5          OOF rows=10,000


In [4]:
reference_name = "CatBoost"
reference_experiment = experiments[reference_name]

reference_oof = reference_experiment.oof_predictions.reset_index(drop=True)

reference_test = reference_experiment.test_predictions.reset_index(drop=True)

required_oof_columns = {
    "row_index",
    "fold",
    "target",
    "probability",
}

required_test_columns = {
    "row_index",
    "probability",
}

for model_name, experiment in experiments.items():
    oof_predictions = experiment.oof_predictions.reset_index(drop=True)

    test_predictions = experiment.test_predictions.reset_index(drop=True)

    missing_oof_columns = required_oof_columns - set(oof_predictions.columns)

    missing_test_columns = required_test_columns - set(test_predictions.columns)

    if missing_oof_columns:
        raise ValueError(
            f"{model_name}: missing OOF columns {sorted(missing_oof_columns)}"
        )

    if missing_test_columns:
        raise ValueError(
            f"{model_name}: missing test columns {sorted(missing_test_columns)}"
        )

    if not reference_oof["row_index"].equals(oof_predictions["row_index"]):
        raise ValueError(f"{model_name}: OOF row indices are not aligned.")

    if not reference_oof["fold"].equals(oof_predictions["fold"]):
        raise ValueError(f"{model_name}: CV fold assignments are not aligned.")

    if not reference_oof["target"].equals(oof_predictions["target"]):
        raise ValueError(f"{model_name}: OOF target values are not aligned.")

    if not reference_test["row_index"].equals(test_predictions["row_index"]):
        raise ValueError(f"{model_name}: test row indices are not aligned.")

print("All baseline experiments are fully aligned.")

All baseline experiments are fully aligned.


In [5]:
comparison_records = []
threshold_diagnostics = {}

for model_name, experiment in experiments.items():
    metrics = experiment.result.metrics
    oof_predictions = experiment.oof_predictions

    cross_fitted_result = evaluate_cross_fitted_thresholds(
        oof_predictions=oof_predictions,
    )

    plateau_result = summarize_threshold_plateau(
        y_true=oof_predictions["target"],
        probabilities=(oof_predictions["probability"].to_numpy()),
        tolerance=0.001,
    )

    cross_fitted_threshold_mean = cross_fitted_result.fold_metrics[
        "threshold_cross_fitted"
    ].mean()

    cross_fitted_threshold_std = cross_fitted_result.fold_metrics[
        "threshold_cross_fitted"
    ].std(ddof=1)

    threshold_diagnostics[model_name] = {
        "cross_fitted": cross_fitted_result,
        "plateau": plateau_result,
    }

    comparison_records.append(
        {
            "model": model_name,
            "balanced_accuracy_default": (metrics["balanced_accuracy"]),
            "optimized_threshold_oof": (metrics["optimized_threshold_oof"]),
            "balanced_accuracy_optimized_oof": (
                metrics["optimized_balanced_accuracy_oof"]
            ),
            "balanced_accuracy_cross_fitted": (cross_fitted_result.balanced_accuracy),
            "threshold_cross_fitted_mean": (cross_fitted_threshold_mean),
            "threshold_cross_fitted_std": (cross_fitted_threshold_std),
            "plateau_lower": (plateau_result.lower_threshold),
            "plateau_upper": (plateau_result.upper_threshold),
            "roc_auc": metrics["roc_auc"],
            "average_precision": (metrics["average_precision"]),
            "log_loss": metrics["log_loss"],
            "training_time_seconds": (experiment.result.training_time_seconds),
        }
    )

model_comparison = pd.DataFrame(comparison_records)

model_comparison["rank_cross_fitted_ba"] = (
    model_comparison["balanced_accuracy_cross_fitted"]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

model_comparison = model_comparison.sort_values(
    "balanced_accuracy_cross_fitted",
    ascending=False,
).reset_index(drop=True)

display(
    model_comparison.style.format(
        {
            "balanced_accuracy_default": "{:.4f}",
            "optimized_threshold_oof": "{:.3f}",
            "balanced_accuracy_optimized_oof": "{:.4f}",
            "balanced_accuracy_cross_fitted": "{:.4f}",
            "threshold_cross_fitted_mean": "{:.3f}",
            "threshold_cross_fitted_std": "{:.4f}",
            "plateau_lower": "{:.3f}",
            "plateau_upper": "{:.3f}",
            "roc_auc": "{:.4f}",
            "average_precision": "{:.4f}",
            "log_loss": "{:.4f}",
            "training_time_seconds": "{:.2f}",
        }
    )
)

,model,balanced_accuracy_default,optimized_threshold_oof,balanced_accuracy_optimized_oof,balanced_accuracy_cross_fitted,threshold_cross_fitted_mean,threshold_cross_fitted_std,plateau_lower,plateau_upper,roc_auc,average_precision,log_loss,training_time_seconds,rank_cross_fitted_ba
0,CatBoost,0.7940,0.174,0.8946,0.8926,0.176,0.0035,0.168,0.176,0.9620,0.8247,0.1601,375.47,1
1,XGBoost,0.7846,0.041,0.8910,0.8907,0.041,0.0005,0.039,0.041,0.9603,0.8161,0.1798,61.13,2
2,LightGBM,0.7634,0.128,0.8813,0.8807,0.129,0.0018,0.126,0.132,0.9537,0.7841,0.1797,11.37,3


## OOF Prediction Agreement

This section measures the similarity of model probability estimates and the disagreement between thresholded predictions.

High probability correlation indicates that models rank observations similarly. Prediction disagreement reveals whether one model can correct errors made by another.

In [8]:
MODEL_NAMES = [
    "CatBoost",
    "XGBoost",
    "LightGBM",
]

oof_probability_frame = pd.DataFrame(
    {
        "row_index": reference_oof["row_index"].to_numpy(),
        "fold": reference_oof["fold"].to_numpy(),
        "target": reference_oof["target"].to_numpy(),
        **{
            model_name: experiments[model_name]
            .oof_predictions["probability"]
            .to_numpy()
            for model_name in MODEL_NAMES
        },
    }
)

display(oof_probability_frame.head())

,row_index,fold,target,CatBoost,XGBoost,LightGBM
0,0,1,0,0.000487,0.000039,0.014142
1,1,3,0,0.000232,0.000111,0.008944
2,2,5,0,0.001535,0.011831,0.015474
3,3,4,0,0.009162,0.001538,0.014105
4,4,4,0,0.024330,0.000835,0.031922


In [9]:
pearson_correlation = oof_probability_frame[MODEL_NAMES].corr(method="pearson")

spearman_correlation = oof_probability_frame[MODEL_NAMES].corr(method="spearman")

print("Pearson correlation")
display(pearson_correlation.style.format("{:.4f}"))

print("Spearman rank correlation")
display(spearman_correlation.style.format("{:.4f}"))

Pearson correlation


,CatBoost,XGBoost,LightGBM
CatBoost,1.0000,0.9107,0.9139
XGBoost,0.9107,1.0000,0.9352
LightGBM,0.9139,0.9352,1.0000


Spearman rank correlation


,CatBoost,XGBoost,LightGBM
CatBoost,1.0000,0.9400,0.8364
XGBoost,0.9400,1.0000,0.8273
LightGBM,0.8364,0.8273,1.0000


In [10]:
optimized_thresholds = {
    model_name: float(experiments[model_name].result.metrics["optimized_threshold_oof"])
    for model_name in MODEL_NAMES
}

thresholded_predictions = pd.DataFrame(
    {
        model_name: (
            oof_probability_frame[model_name] >= optimized_thresholds[model_name]
        ).astype(int)
        for model_name in MODEL_NAMES
    }
)

target = oof_probability_frame["target"].to_numpy()

disagreement_records = []

for left_model, right_model in combinations(
    MODEL_NAMES,
    2,
):
    left_predictions = thresholded_predictions[left_model].to_numpy()

    right_predictions = thresholded_predictions[right_model].to_numpy()

    left_correct = left_predictions == target
    right_correct = right_predictions == target

    disagreement_records.append(
        {
            "model_left": left_model,
            "model_right": right_model,
            "prediction_disagreement": float(
                np.mean(left_predictions != right_predictions)
            ),
            "left_only_correct": float(np.mean(left_correct & ~right_correct)),
            "right_only_correct": float(np.mean(~left_correct & right_correct)),
            "both_wrong": float(np.mean(~left_correct & ~right_correct)),
        }
    )

prediction_disagreement = pd.DataFrame(disagreement_records)

display(
    prediction_disagreement.style.format(
        {
            "prediction_disagreement": "{:.2%}",
            "left_only_correct": "{:.2%}",
            "right_only_correct": "{:.2%}",
            "both_wrong": "{:.2%}",
        }
    )
)

,model_left,model_right,prediction_disagreement,left_only_correct,right_only_correct,both_wrong
0,CatBoost,XGBoost,5.89%,3.57%,2.32%,8.71%
1,CatBoost,LightGBM,6.73%,4.32%,2.41%,8.62%
2,XGBoost,LightGBM,5.58%,3.12%,2.46%,9.82%


## Fixed Probability Blends

Several predefined blends are evaluated before any data-driven weight optimization.

For each blend, the decision threshold is selected using the same cross-fitted procedure used for the standalone models. This avoids evaluating the threshold on the same fold where it was selected.

In [11]:
BLEND_SPECS = {
    "CatBoost 50% + XGBoost 50%": {
        "CatBoost": 0.50,
        "XGBoost": 0.50,
    },
    "CatBoost 60% + XGBoost 40%": {
        "CatBoost": 0.60,
        "XGBoost": 0.40,
    },
    "CatBoost 40% + XGBoost 60%": {
        "CatBoost": 0.40,
        "XGBoost": 0.60,
    },
    "CatBoost 50% + LightGBM 50%": {
        "CatBoost": 0.50,
        "LightGBM": 0.50,
    },
    "XGBoost 50% + LightGBM 50%": {
        "XGBoost": 0.50,
        "LightGBM": 0.50,
    },
    "Equal three-model blend": {
        "CatBoost": 1 / 3,
        "XGBoost": 1 / 3,
        "LightGBM": 1 / 3,
    },
}

In [12]:
blend_records = []
blend_oof_predictions = {}

for blend_name, weights in BLEND_SPECS.items():
    blended_probabilities = np.zeros(
        len(oof_probability_frame),
        dtype=float,
    )

    for model_name, weight in weights.items():
        blended_probabilities += weight * oof_probability_frame[model_name].to_numpy()

    blend_predictions = pd.DataFrame(
        {
            "row_index": (oof_probability_frame["row_index"].to_numpy()),
            "fold": (oof_probability_frame["fold"].to_numpy()),
            "target": (oof_probability_frame["target"].to_numpy()),
            "probability": blended_probabilities,
        }
    )

    cross_fitted_result = evaluate_cross_fitted_thresholds(
        oof_predictions=blend_predictions,
    )

    global_optimization = optimize_balanced_accuracy_threshold(
        y_true=blend_predictions["target"],
        probabilities=blended_probabilities,
    )

    cross_fitted_thresholds = cross_fitted_result.fold_metrics["threshold_cross_fitted"]

    blend_records.append(
        {
            "blend": blend_name,
            "balanced_accuracy_optimized_oof": (global_optimization.balanced_accuracy),
            "balanced_accuracy_cross_fitted": (cross_fitted_result.balanced_accuracy),
            "optimized_threshold_oof": (global_optimization.threshold),
            "threshold_cross_fitted_mean": (cross_fitted_thresholds.mean()),
            "threshold_cross_fitted_std": (cross_fitted_thresholds.std(ddof=1)),
            "roc_auc": roc_auc_score(
                blend_predictions["target"],
                blended_probabilities,
            ),
            "average_precision": (
                average_precision_score(
                    blend_predictions["target"],
                    blended_probabilities,
                )
            ),
            "log_loss": log_loss(
                blend_predictions["target"],
                blended_probabilities,
                labels=[0, 1],
            ),
        }
    )

    blend_oof_predictions[blend_name] = blend_predictions

fixed_blend_results = pd.DataFrame(blend_records)

In [13]:
catboost_cross_fitted_ba = float(
    model_comparison.loc[
        model_comparison["model"] == "CatBoost",
        "balanced_accuracy_cross_fitted",
    ].iloc[0]
)

fixed_blend_results["delta_vs_catboost"] = (
    fixed_blend_results["balanced_accuracy_cross_fitted"] - catboost_cross_fitted_ba
)

fixed_blend_results = fixed_blend_results.sort_values(
    "balanced_accuracy_cross_fitted",
    ascending=False,
).reset_index(drop=True)

display(
    fixed_blend_results.style.format(
        {
            "balanced_accuracy_optimized_oof": "{:.4f}",
            "balanced_accuracy_cross_fitted": "{:.4f}",
            "optimized_threshold_oof": "{:.3f}",
            "threshold_cross_fitted_mean": "{:.3f}",
            "threshold_cross_fitted_std": "{:.4f}",
            "roc_auc": "{:.4f}",
            "average_precision": "{:.4f}",
            "log_loss": "{:.4f}",
            "delta_vs_catboost": "{:+.4f}",
        }
    )
)

,blend,balanced_accuracy_optimized_oof,balanced_accuracy_cross_fitted,optimized_threshold_oof,threshold_cross_fitted_mean,threshold_cross_fitted_std,roc_auc,average_precision,log_loss,delta_vs_catboost
0,CatBoost 50% + XGBoost 50%,0.8988,0.8988,0.130,0.130,0.0000,0.9644,0.8346,0.1560,+0.0062
1,CatBoost 40% + XGBoost 60%,0.8976,0.8962,0.117,0.119,0.0048,0.9642,0.8329,0.1572,+0.0036
2,CatBoost 50% + LightGBM 50%,0.8961,0.8944,0.150,0.153,0.0053,0.9629,0.8255,0.1636,+0.0017
3,CatBoost 60% + XGBoost 40%,0.8968,0.8940,0.120,0.129,0.0124,0.9644,0.8356,0.1556,+0.0014
4,Equal three-model blend,0.8962,0.8922,0.114,0.116,0.0119,0.9638,0.8293,0.1598,-0.0004
5,XGBoost 50% + LightGBM 50%,0.8908,0.8888,0.084,0.083,0.0048,0.9599,0.8118,0.1685,-0.0039


## Cross-Fitted Blend Weight Selection

For each holdout fold, the CatBoost/XGBoost blend weight and decision threshold are selected using only the remaining folds.

The selected configuration is then evaluated on the untouched holdout fold. This estimates the full model-selection procedure without using holdout labels during weight or threshold selection.

In [ ]:
catboost_probabilities = oof_probability_frame["CatBoost"].to_numpy()

xgboost_probabilities = oof_probability_frame["XGBoost"].to_numpy()

targets = oof_probability_frame["target"].to_numpy()
folds = oof_probability_frame["fold"].to_numpy()

catboost_weight_grid = np.round(
    np.linspace(0.0, 1.0, 21),
    2,
)

cross_fitted_blend_probabilities = np.zeros(
    len(oof_probability_frame),
    dtype=float,
)

cross_fitted_blend_predictions = np.zeros(
    len(oof_probability_frame),
    dtype=int,
)

blend_selection_records = []

for holdout_fold in sorted(np.unique(folds)):
    training_mask = folds != holdout_fold
    holdout_mask = folds == holdout_fold

    candidate_records = []

    for catboost_weight in catboost_weight_grid:
        xgboost_weight = 1.0 - catboost_weight

        training_blend = (
            catboost_weight * catboost_probabilities[training_mask]
            + xgboost_weight * xgboost_probabilities[training_mask]
        )

        threshold_result = optimize_balanced_accuracy_threshold(
            y_true=targets[training_mask],
            probabilities=training_blend,
        )

        candidate_records.append(
            {
                "catboost_weight": catboost_weight,
                "xgboost_weight": xgboost_weight,
                "threshold": threshold_result.threshold,
                "training_balanced_accuracy": (threshold_result.balanced_accuracy),
                "distance_from_equal_weight": abs(catboost_weight - 0.5),
            }
        )

    candidate_frame = pd.DataFrame(candidate_records)

    best_candidate = candidate_frame.sort_values(
        [
            "training_balanced_accuracy",
            "distance_from_equal_weight",
            "catboost_weight",
        ],
        ascending=[False, True, False],
    ).iloc[0]

    selected_catboost_weight = float(best_candidate["catboost_weight"])
    selected_xgboost_weight = float(best_candidate["xgboost_weight"])
    selected_threshold = float(best_candidate["threshold"])

    holdout_probabilities = (
        selected_catboost_weight * catboost_probabilities[holdout_mask]
        + selected_xgboost_weight * xgboost_probabilities[holdout_mask]
    )

    holdout_predictions = (holdout_probabilities >= selected_threshold).astype(int)

    holdout_balanced_accuracy = balanced_accuracy_score(
        targets[holdout_mask],
        holdout_predictions,
    )

    cross_fitted_blend_probabilities[holdout_mask] = holdout_probabilities

    cross_fitted_blend_predictions[holdout_mask] = holdout_predictions

    blend_selection_records.append(
        {
            "fold": int(holdout_fold),
            "catboost_weight": (selected_catboost_weight),
            "xgboost_weight": (selected_xgboost_weight),
            "threshold": selected_threshold,
            "training_balanced_accuracy": float(
                best_candidate["training_balanced_accuracy"]
            ),
            "holdout_balanced_accuracy": (holdout_balanced_accuracy),
        }
    )

cross_fitted_blend_selection = pd.DataFrame(blend_selection_records)

cross_fitted_blend_balanced_accuracy = balanced_accuracy_score(
    targets,
    cross_fitted_blend_predictions,
)

display(
    cross_fitted_blend_selection.style.format(
        {
            "catboost_weight": "{:.2f}",
            "xgboost_weight": "{:.2f}",
            "threshold": "{:.3f}",
            "training_balanced_accuracy": "{:.4f}",
            "holdout_balanced_accuracy": "{:.4f}",
        }
    )
)

print(f"Cross-fitted selected-weight BA: {cross_fitted_blend_balanced_accuracy:.4f}")
print(
    "Mean selected CatBoost weight: "
    f"{cross_fitted_blend_selection['catboost_weight'].mean():.3f}"
)
print(
    "Selected CatBoost weight std:  "
    f"{cross_fitted_blend_selection['catboost_weight'].std(ddof=1):.3f}"
)
print(
    "Mean selected threshold:       "
    f"{cross_fitted_blend_selection['threshold'].mean():.3f}"
)

,fold,catboost_weight,xgboost_weight,threshold,training_balanced_accuracy,holdout_balanced_accuracy
0,1,0.50,0.50,0.130,0.8958,0.9108
1,2,0.50,0.50,0.130,0.9022,0.8853
2,3,0.50,0.50,0.130,0.8990,0.8980
3,4,0.45,0.55,0.127,0.8974,0.9005
4,5,0.50,0.50,0.130,0.9000,0.8940


Cross-fitted selected-weight BA: 0.8977
Mean selected CatBoost weight: 0.490
Selected CatBoost weight std:  0.022
Mean selected threshold:       0.129


In [15]:
selected_blend_name = "CatBoost 50% + XGBoost 50%"

selected_blend_oof = blend_oof_predictions[selected_blend_name]

selected_blend_plateau = summarize_threshold_plateau(
    y_true=selected_blend_oof["target"],
    probabilities=selected_blend_oof["probability"].to_numpy(),
    tolerance=0.001,
)

selected_blend_result = fixed_blend_results.loc[
    fixed_blend_results["blend"] == selected_blend_name
].iloc[0]

selected_blend_threshold = float(selected_blend_result["optimized_threshold_oof"])

selected_threshold_inside_plateau = (
    selected_blend_plateau.lower_threshold
    <= selected_blend_threshold
    <= selected_blend_plateau.upper_threshold
)

selected_blend_summary = pd.DataFrame(
    [
        {
            "blend": selected_blend_name,
            "cross_fitted_balanced_accuracy": (
                selected_blend_result["balanced_accuracy_cross_fitted"]
            ),
            "optimized_oof_threshold": (selected_blend_threshold),
            "plateau_lower": (selected_blend_plateau.lower_threshold),
            "plateau_upper": (selected_blend_plateau.upper_threshold),
            "plateau_width": (selected_blend_plateau.width),
            "threshold_inside_plateau": (selected_threshold_inside_plateau),
        }
    ]
)

display(
    selected_blend_summary.style.format(
        {
            "cross_fitted_balanced_accuracy": "{:.4f}",
            "optimized_oof_threshold": "{:.3f}",
            "plateau_lower": "{:.3f}",
            "plateau_upper": "{:.3f}",
            "plateau_width": "{:.3f}",
        }
    )
)

,blend,cross_fitted_balanced_accuracy,optimized_oof_threshold,plateau_lower,plateau_upper,plateau_width,threshold_inside_plateau
0,CatBoost 50% + XGBoost 50%,0.8988,0.130,0.127,0.131,0.004,True


## Selected Blend Test Predictions

The final comparison-stage candidate uses an equal-weight average of CatBoost and XGBoost probabilities.

The deployment threshold is fitted using all available OOF predictions. Cross-fitted evaluation is used to validate the selection procedure, while the global OOF threshold is retained for test predictions.

In [16]:
catboost_test = experiments["CatBoost"].test_predictions.reset_index(drop=True)

xgboost_test = experiments["XGBoost"].test_predictions.reset_index(drop=True)

if not catboost_test["row_index"].equals(xgboost_test["row_index"]):
    raise ValueError("CatBoost and XGBoost test rows are not aligned.")

selected_test_probabilities = (
    0.5 * catboost_test["probability"].to_numpy()
    + 0.5 * xgboost_test["probability"].to_numpy()
)

selected_test_predictions = (
    selected_test_probabilities >= selected_blend_threshold
).astype(int)

selected_blend_test_predictions = pd.DataFrame(
    {
        "row_index": catboost_test["row_index"].to_numpy(),
        "probability": selected_test_probabilities,
        "prediction": selected_test_predictions,
    }
)

print(f"Selected blend threshold: {selected_blend_threshold:.3f}")
print(f"Predicted positive rate:  {selected_test_predictions.mean():.2%}")
print(f"Predicted positive rows:  {selected_test_predictions.sum():,}")
print(f"Total test rows:          {len(selected_test_predictions):,}")

display(selected_blend_test_predictions.head())

Selected blend threshold: 0.130
Predicted positive rate:  20.36%
Predicted positive rows:  509
Total test rows:          2,500


,row_index,probability,prediction
0,0,0.001349,0
1,1,0.030158,0
2,2,0.005894,0
3,3,0.009900,0
4,4,0.004379,0


In [17]:
COMPARISON_ID = "comparison_v0_raw_minimal"
COMPARISON_DIR = EXPERIMENTS_DIR / COMPARISON_ID

COMPARISON_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

model_comparison.to_csv(
    COMPARISON_DIR / "model_comparison.csv",
    index=False,
)

prediction_disagreement.to_csv(
    COMPARISON_DIR / "prediction_disagreement.csv",
    index=False,
)

fixed_blend_results.to_csv(
    COMPARISON_DIR / "fixed_blend_results.csv",
    index=False,
)

cross_fitted_blend_selection.to_csv(
    COMPARISON_DIR / "cross_fitted_blend_selection.csv",
    index=False,
)

selected_blend_oof.to_parquet(
    COMPARISON_DIR / "selected_blend_oof_predictions.parquet",
    index=False,
)

selected_blend_test_predictions.to_parquet(
    COMPARISON_DIR / "selected_blend_test_predictions.parquet",
    index=False,
)

print(f"Comparison artifacts saved to: {COMPARISON_DIR}")

Comparison artifacts saved to: C:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\artifacts\experiments\comparison_v0_raw_minimal


## Conclusion

Three gradient-boosting baselines were evaluated using identical five-fold stratified cross-validation splits and aligned out-of-fold predictions.

CatBoost was the strongest standalone model with a cross-fitted Balanced Accuracy of approximately **0.893**. XGBoost followed closely at approximately **0.891**, while LightGBM reached approximately **0.881**.

Although CatBoost and XGBoost produced highly correlated predictions, they disagreed on approximately **5.9%** of observations and corrected different subsets of errors. This complementarity resulted in a meaningful ensemble improvement.

The equal-weight CatBoost/XGBoost blend achieved:

- cross-fitted Balanced Accuracy of approximately **0.899**
- ROC-AUC of approximately **0.964**
- Average Precision of approximately **0.835**
- an optimized OOF threshold of **0.130**

Cross-fitted weight selection chose an average CatBoost weight of approximately **0.49**, with four of five folds selecting the exact 50/50 allocation. The tuned-weight procedure produced a slightly lower cross-fitted score than the fixed equal-weight blend.

Therefore, the selected comparison-stage model is:

- **50% CatBoost**
- **50% XGBoost**
- **decision threshold: 0.130**

Further hyperparameter tuning is deferred. The next experimental stage will focus on alternative data-processing and feature-engineering versions, since changes to the input representation are more likely to produce substantial improvements than fine adjustments to the current baseline parameters.